In [2]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

def calculate_solar_forecast(lat, lon, kwp, tilt, azimuth, yield_factor=0.75):
    # 1. Create a 15-minute time range for the next 24 hours
    start_time = datetime.now().replace(minute=0, second=0, microsecond=0)
    times = [start_time + timedelta(minutes=15*i) for i in range(96)]
    
    # 2. Basic Constants
    solar_constant = 1000  # Standard Test Condition (W/m2)
    day_of_year = datetime.now().timetuple().tm_yday
    
    # 3. Calculate Solar Position (Simplified)
    # Declination angle (approximate for the day)
    declination = 23.45 * np.sin(np.radians(360/365 * (day_of_year - 81)))
    
    results = []
    for t in times:
        # Hour angle: 15 degrees per hour from solar noon (12:00)
        hour_angle = 15 * (t.hour + t.minute/60 - 12)
        
        # Solar Zenith Angle (angle from directly overhead)
        cos_zenith = (np.sin(np.radians(lat)) * np.sin(np.radians(declination)) + 
                      np.cos(np.radians(lat)) * np.cos(np.radians(declination)) * 
                      np.cos(np.radians(hour_angle)))
        zenith = np.degrees(np.arccos(np.clip(cos_zenith, -1, 1)))
        
        # If sun is below horizon, output is 0
        if zenith > 90:
            power_output = 0
        else:
            # 4. Calculate Incident Angle on Tilted Surface
            # 0 azimuth = South, 90 = West, -90 = East
            # This formula finds how 'directly' the sun hits the panel
            cos_incidence = (np.cos(np.radians(zenith)) * np.cos(np.radians(tilt)) + 
                             np.sin(np.radians(zenith)) * np.sin(np.radians(tilt)) * 
                             np.cos(np.radians(hour_angle - azimuth)))
            
            # 5. Final Power Calculation (kW)
            # Power = Peak Power * Efficiency * (Actual Irradiance / 1000W/m2)
            # We assume a clear sky irradiance of ~1000W/m2 * cos(zenith)
            irradiance = solar_constant * max(0, cos_incidence)
            power_output = kwp * yield_factor * (irradiance / 1000)
            
        results.append({"Time": t.strftime("%H:%M"), "Power_kW": round(power_output, 3)})

    return pd.DataFrame(results)

# --- Define Your Inputs ---
config = {
    "lat": 51.2,          # Latitude (e.g., London)
    "lon": -0.1,          # Longitude
    "kwp": 10.0,           # System Peak Power (kWp)
    "tilt": 35,           # Panel Tilt Angle (degrees)
    "azimuth": 0,         # Orientation (0=South, -90=East, 90=West)
    "yield_factor": 0.80  # Efficiency factor (losses for inverter, heat, etc.)
}

forecast_df = calculate_solar_forecast(**config)
print(forecast_df.head(20)) # Display first 5 hours




     Time  Power_kW
0   15:00     5.442
1   15:15     5.077
2   15:30     4.683
3   15:45     4.259
4   16:00     3.808
5   16:15     3.332
6   16:30     2.833
7   16:45     2.314
8   17:00     1.778
9   17:15     1.229
10  17:30     0.669
11  17:45     0.000
12  18:00     0.000
13  18:15     0.000
14  18:30     0.000
15  18:45     0.000
16  19:00     0.000
17  19:15     0.000
18  19:30     0.000
19  19:45     0.000


In [7]:
import requests
import pandas as pd

def get_solar_weather(lat, lon, kwp, tilt_angle):
    base_url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": "direct_radiation,diffuse_radiation,temperature_2m",
        "forecast_days": 1,
        "timezone": "auto",
    }

    response = requests.get(base_url, params=params, timeout=30)
    response.raise_for_status()
    payload = response.json()
    if "hourly" not in payload:
        raise ValueError("Open-Meteo response has no 'hourly' data")
    data = payload["hourly"]

    df = pd.DataFrame(data)
    df['time'] = pd.to_datetime(df['time'])

    # Constants
    system_loss = 0.85  # 15% loss for dust, inverter, wires
    temp_coeff = -0.004  # Panels lose 0.4% efficiency per degree above 25C

    # Simplified projection: Adjusting direct radiation by tilt
    # A rough estimate: radiation * cos(tilt_difference)
    # For a perfect calculation, you'd use the sun's position vs panel angle
    tilt_correction = 0.9  # Assuming panel is well-aligned

    def calculate_output(row):
        # Calculate effective irradiance
        effective_irradiance = (row['direct_radiation'] * tilt_correction) + row['diffuse_radiation']

        # Base Power: (Irradiance / 1000W/m2) * kWp
        power = (effective_irradiance / 1000) * kwp * system_loss

        # Temperature Correction
        if row['temperature_2m'] > 25:
            power *= (1 + (row['temperature_2m'] - 25) * temp_coeff)

        return max(0, power)

    df['predicted_kw'] = df.apply(calculate_output, axis=1)
    return df[['time', 'predicted_kw', 'temperature_2m']]

# Example for a 5kWp system
forecast = get_solar_weather(51.2, 0.00, 10.0, 35)
print(forecast.to_string())


                  time  predicted_kw  temperature_2m
0  2026-03-09 00:00:00       0.00000             7.1
1  2026-03-09 01:00:00       0.00000             6.9
2  2026-03-09 02:00:00       0.00000             6.6
3  2026-03-09 03:00:00       0.00000             6.8
4  2026-03-09 04:00:00       0.00000             6.8
5  2026-03-09 05:00:00       0.00000             6.8
6  2026-03-09 06:00:00       0.00000             6.8
7  2026-03-09 07:00:00       0.03400             6.9
8  2026-03-09 08:00:00       0.40800             7.3
9  2026-03-09 09:00:00       0.80750             7.8
10 2026-03-09 10:00:00       1.13900             8.7
11 2026-03-09 11:00:00       2.15900            10.2
12 2026-03-09 12:00:00       1.93375            11.3
13 2026-03-09 13:00:00       2.05275            12.0
14 2026-03-09 14:00:00       1.63540            12.7
15 2026-03-09 15:00:00       1.64305            12.8
16 2026-03-09 16:00:00       1.38295            12.2
17 2026-03-09 17:00:00       1.02340          

In [8]:
response

NameError: name 'response' is not defined